In [1]:
import sys
!{sys.executable} -m pip install plotly
import pandas as pd
import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
import plotly.graph_objects as go

df = pd.read_csv("data/dashboard_incidents.csv")

print(f"Loaded {len(df):,} incidents")
df.head()

Loaded 4,992 incidents


,id,match_id,period,minute,second,player,team,home_team,away_team,referee,...,play_pattern,x,y,referee_call,model_pred,p_no_foul,p_foul,p_yellow,p_red,surprise
0,f95afad1-fa3a-4aa3-8fc9-3e32d7d1d12b,3754068,2,81,27,Jordan Amavi,Aston Villa,Aston Villa,West Bromwich Albion,Martin Atkinson,...,From Throw In,2.2,8.4,yellow,no_foul,0.921068,0.075921,0.002898,0.000113,2.665805
1,6a9291e6-44b0-4476-a840-7674983fab65,3754110,1,7,22,Carl Jenkinson,West Ham United,West Ham United,Manchester City,Craig Pawson,...,Regular Play,10.0,58.1,foul,no_foul,0.958903,0.034866,0.006135,0.000096,2.272405
2,1c81d1e7-af4e-41db-be95-4f6ad4a70011,3754166,2,92,35,José Leonardo Ulloa,Leicester City,Everton,Leicester City,Jonathan Moss,...,From Free Kick,106.4,35.9,no_foul,foul,0.064321,0.893336,0.039967,0.002376,2.264032
3,0d54feaa-07b4-4883-8a11-7d1c23e83a02,3754054,2,67,49,Sebastián Coates Nión,Sunderland,Manchester City,Sunderland,Anthony Taylor,...,Regular Play,16.8,48.0,foul,no_foul,0.904434,0.042164,0.028866,0.024537,2.082340
4,88211324-1a1e-4b7e-b6d3-8ca802162aea,3754269,1,2,26,Ashley Williams,Swansea City,Swansea City,Stoke City,Robert Madley,...,Regular Play,7.8,26.2,foul,no_foul,0.955748,0.042610,0.001625,0.000016,2.071819


In [14]:

def make_probability_chart(selected_incident):
    # 1. Define the four outcomes and read their probabilities
    outcomes = ["No foul", "Foul", "Yellow card", "Red card"]
    probability_columns = ["p_no_foul", "p_foul", "p_yellow", "p_red"]

    probabilities = [
        float(selected_incident[col])
        for col in probability_columns
    ]

    # Convert probabilities to percentages for display
    percentages = [p * 100 for p in probabilities]

    # 2. Normalize the referee's decision to match our outcome labels
    def normalize_call(value):
        value = str(value).strip().lower().replace("_", " ")
        value = " ".join(value.split())

        aliases = {
            "no foul": "No foul",
            "no-foul": "No foul",
            "foul": "Foul",
            "yellow": "Yellow card",
            "yellow card": "Yellow card",
            "red": "Red card",
            "red card": "Red card",
        }
        return aliases.get(value, value.title())

    actual_call = normalize_call(selected_incident["referee_call"])

    # 3. Highlight the actual referee decision
    colors = [
        "#2E8B57" if outcome == actual_call else "#9EADBD"
        for outcome in outcomes
    ]

    # 4. Build the horizontal bar chart
    fig = go.Figure(
        data=[
            go.Bar(
                x=percentages,
                y=outcomes,
                orientation="h",
                marker_color=colors,
                text=[f"{p:.1f}%" for p in percentages],
                textposition="outside",
                cliponaxis=False,
                customdata=[
                    "Actual referee decision"
                    if outcome == actual_call
                    else "Model probability"
                    for outcome in outcomes
                ],
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Probability: %{x:.2f}%<br>"
                    "%{customdata}"
                    "<extra></extra>"
                ),
            )
        ]
    )

    fig.update_layout(
        title="Model probability of each decision",
        xaxis_title="Probability (%)",
        yaxis_title="Decision",
        xaxis=dict(range=[0, max(105, max(percentages, default=0) * 1.15)]),
        yaxis=dict(autorange="reversed"),
        template="plotly_white",
        height=350,
        margin=dict(l=110, r=50, t=65, b=55),
        showlegend=False,
    )

    return fig


def make_pitch_plot(filtered_df, selected_incident):
    # 1. Define colors for each referee decision
    def normalize_call(value):
        value = str(value).strip().lower().replace("_", " ")
        value = " ".join(value.split())

        aliases = {
            "no foul": "No foul",
            "no-foul": "No foul",
            "foul": "Foul",
            "yellow": "Yellow card",
            "yellow card": "Yellow card",
            "red": "Red card",
            "red card": "Red card",
        }
        return aliases.get(value, value.title())

    decision_colors = {
        "No foul": "#87CEEB",
        "Foul": "#E5B93F",
        "Yellow card": "#E67E22",
        "Red card": "#D62728",
    }

    # Work with a copy so the original DataFrame is unchanged
    plot_df = filtered_df.copy()

    if not plot_df.empty:
        plot_df["decision_label"] = plot_df["referee_call"].map(normalize_call)

    # 2. Draw the soccer pitch using StatsBomb-style coordinates
    fig = go.Figure()

    pitch_color = "#167C48"
    line_color = "white"

    # Pitch background
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=120, y1=80,
        fillcolor=pitch_color,
        line=dict(color=line_color, width=2),
        layer="below",
    )

    # Halfway line
    fig.add_shape(
        type="line",
        x0=60, y0=0, x1=60, y1=80,
        line=dict(color=line_color, width=2),
    )

    # Center circle and center spot
    fig.add_shape(
        type="circle",
        x0=50, y0=30, x1=70, y1=50,
        line=dict(color=line_color, width=2),
    )
    fig.add_trace(
        go.Scatter(
            x=[60], y=[40],
            mode="markers",
            marker=dict(color=line_color, size=5),
            hoverinfo="skip",
            showlegend=False,
        )
    )

    # Penalty areas
    for x0, x1 in [(0, 18), (102, 120)]:
        fig.add_shape(
            type="rect",
            x0=x0, y0=18, x1=x1, y1=62,
            line=dict(color=line_color, width=2),
            fillcolor="rgba(0,0,0,0)",
        )

    # Six-yard boxes
    for x0, x1 in [(0, 6), (114, 120)]:
        fig.add_shape(
            type="rect",
            x0=x0, y0=30, x1=x1, y1=50,
            line=dict(color=line_color, width=2),
            fillcolor="rgba(0,0,0,0)",
        )

    # Penalty spots
    fig.add_trace(
        go.Scatter(
            x=[12, 108], y=[40, 40],
            mode="markers",
            marker=dict(color=line_color, size=4),
            hoverinfo="skip",
            showlegend=False,
        )
    )

    # 3. Plot filtered incidents, grouped by referee decision
    for decision, color in decision_colors.items():
        group = plot_df[plot_df["decision_label"] == decision]

        # Keep a trace for each decision, even if its group is empty
        fig.add_trace(
            go.Scatter(
                x=group["x"],
                y=group["y"],
                mode="markers",
                name=decision,
                marker=dict(
                    size=7,
                    color=color,
                    opacity=0.85,
                    line=dict(color="white", width=0.5),
                ),
                customdata=group[
                    ["player", "minute", "referee_call", "surprise"]
                ].to_numpy(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Minute: %{customdata[1]}<br>"
                    "Referee call: %{customdata[2]}<br>"
                    "Surprise score: %{customdata[3]:.3f}<br>"
                    "Location: (%{x:.1f}, %{y:.1f})"
                    "<extra></extra>"
                ),
            )
        )

    # 4. Highlight the selected incident on top of the other markers
    if (
        pd.notna(selected_incident.get("x"))
        and pd.notna(selected_incident.get("y"))
    ):
        player = selected_incident.get("player", "Unknown")
        minute = selected_incident.get("minute", "Unknown")
        referee_call = selected_incident.get("referee_call", "Unknown")
        surprise = selected_incident.get("surprise", float("nan"))

        selected_color = decision_colors.get(
            normalize_call(referee_call), "#FFFFFF"
        )

        fig.add_trace(
            go.Scatter(
                x=[selected_incident["x"]],
                y=[selected_incident["y"]],
                mode="markers",
                name="Selected incident",
                marker=dict(
                    size=17,
                    color=selected_color,
                    symbol="circle",
                    line=dict(color="#111111", width=3),
                ),
                hovertemplate=(
                    "<b>SELECTED INCIDENT</b><br>"
                    f"Player: {player}<br>"
                    f"Minute: {minute}<br>"
                    f"Referee call: {referee_call}<br>"
                    f"Surprise score: {surprise}<br>"
                    "Location: (%{x:.1f}, %{y:.1f})"
                    "<extra></extra>"
                ),
            )
        )

    # 5. Format the pitch and axes
    fig.update_layout(
        title="Incident location on pitch",
        template="plotly_white",
        height=550,
        xaxis=dict(
            range=[-2, 122],
            title="Length",
            showgrid=False,
            zeroline=False,
            constrain="domain",
        ),
        yaxis=dict(
            range=[-2, 82],
            title="Width",
            showgrid=False,
            zeroline=False,
            scaleanchor="x",
            scaleratio=1,
        ),
        plot_bgcolor=pitch_color,
        paper_bgcolor="white",
        legend=dict(
            title="Referee decision",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
        ),
        margin=dict(l=45, r=25, t=100, b=45),
    )

    return fig


In [15]:
import math
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

DATA_PATH = Path("data/dashboard_incidents.csv")

df = pd.read_csv(DATA_PATH)

# Makes a readable match name for the dashboard filters and table.
df["match_label"] = (
    df["home_team"] + " vs " + df["away_team"]
    + " (Match " + df["match_id"].astype(str) + ")"
)

# Finds the model probability assigned to the referee's actual decision.
probability_column = {
    "no_foul": "p_no_foul",
    "foul": "p_foul",
    "yellow": "p_yellow",
    "red": "p_red",
}

df["actual_probability"] = df.apply(
    lambda row: row[probability_column[row["referee_call"]]],
    axis=1
)

print(f"Loaded {len(df):,} incidents")
df.head()

Loaded 4,992 incidents


,id,match_id,period,minute,second,player,team,home_team,away_team,referee,...,y,referee_call,model_pred,p_no_foul,p_foul,p_yellow,p_red,surprise,match_label,actual_probability
0,f95afad1-fa3a-4aa3-8fc9-3e32d7d1d12b,3754068,2,81,27,Jordan Amavi,Aston Villa,Aston Villa,West Bromwich Albion,Martin Atkinson,...,8.4,yellow,no_foul,0.921068,0.075921,0.002898,0.000113,2.665805,Aston Villa vs West Bromwich Albion (Match 375...,0.002898
1,6a9291e6-44b0-4476-a840-7674983fab65,3754110,1,7,22,Carl Jenkinson,West Ham United,West Ham United,Manchester City,Craig Pawson,...,58.1,foul,no_foul,0.958903,0.034866,0.006135,0.000096,2.272405,West Ham United vs Manchester City (Match 3754...,0.034866
2,1c81d1e7-af4e-41db-be95-4f6ad4a70011,3754166,2,92,35,José Leonardo Ulloa,Leicester City,Everton,Leicester City,Jonathan Moss,...,35.9,no_foul,foul,0.064321,0.893336,0.039967,0.002376,2.264032,Everton vs Leicester City (Match 3754166),0.064321
3,0d54feaa-07b4-4883-8a11-7d1c23e83a02,3754054,2,67,49,Sebastián Coates Nión,Sunderland,Manchester City,Sunderland,Anthony Taylor,...,48.0,foul,no_foul,0.904434,0.042164,0.028866,0.024537,2.082340,Manchester City vs Sunderland (Match 3754054),0.042164
4,88211324-1a1e-4b7e-b6d3-8ca802162aea,3754269,1,2,26,Ashley Williams,Swansea City,Swansea City,Stoke City,Robert Madley,...,26.2,foul,no_foul,0.955748,0.042610,0.001625,0.000016,2.071819,Swansea City vs Stoke City (Match 3754269),0.042610


In [16]:
match_dropdown = widgets.Dropdown(
    options=["All matches"] + sorted(df["match_label"].unique().tolist()),
    value="All matches",
    description="Match:",
    layout=widgets.Layout(width="360px")
)

referee_dropdown = widgets.Dropdown(
    options=["All referees"] + sorted(df["referee"].dropna().unique().tolist()),
    value="All referees",
    description="Referee:",
    layout=widgets.Layout(width="300px")
)

decision_dropdown = widgets.Dropdown(
    options=[
        ("All decisions", "all"),
        ("No foul", "no_foul"),
        ("Foul", "foul"),
        ("Yellow card", "yellow"),
        ("Red card", "red"),
    ],
    value="all",
    description="Decision:",
    layout=widgets.Layout(width="230px")
)

surprise_slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=math.ceil(df["surprise"].max() * 10) / 10,
    step=0.1,
    description="Min. surprise:",
    readout_format=".1f",
    layout=widgets.Layout(width="360px")
)

controls = widgets.VBox([
    widgets.HBox([match_dropdown, referee_dropdown]),
    widgets.HBox([decision_dropdown, surprise_slider])
])

#display(controls)

In [17]:
def get_filtered_data():
    filtered = df.copy()

    if match_dropdown.value != "All matches":
        filtered = filtered[
            filtered["match_label"] == match_dropdown.value
        ]

    if referee_dropdown.value != "All referees":
        filtered = filtered[
            filtered["referee"] == referee_dropdown.value
        ]

    if decision_dropdown.value != "all":
        filtered = filtered[
            filtered["referee_call"] == decision_dropdown.value
        ]

    return filtered

In [18]:
summary_output = widgets.Output()
table_output = widgets.Output()

def metric_card(label, value, note=""):
    return widgets.HTML(
        value=f"""
        <div style="
            border: 1px solid #b8c7d9;
            border-radius: 8px;
            padding: 14px;
            width: 220px;
            background: #ffffff;
            color: #172235;
        ">
            <div style="font-size: 12px; color: #334155;">{label}</div>
            <div style="font-size: 26px; font-weight: 700; color: #0f3d69;">
                {value}
            </div>
            <div style="font-size: 11px; color: #475569;">{note}</div>
        </div>
        """
    )


def update_your_dashboard(change=None):
    filtered_df = get_filtered_data()

    # Incidents that meet the surprise threshold, ranked highest first.
    priority_df = filtered_df[
        filtered_df["surprise"] >= surprise_slider.value
    ].sort_values("surprise", ascending=False)

    # Keep these available for the pitch and probability charts later.
    globals()["current_filtered_df"] = filtered_df
    globals()["current_priority_df"] = priority_df

    average_probability = (
        filtered_df["actual_probability"].mean()
        if len(filtered_df) > 0 else 0
    )

    with summary_output:
        clear_output(wait=True)

        display(widgets.HBox([
            metric_card(
                "Incidents analyzed",
                f"{len(filtered_df):,}",
                "After current filters"
            ),
            metric_card(
                "High-priority reviews",
                f"{len(priority_df):,}",
                f"Surprise score ≥ {surprise_slider.value:.1f}"
            ),
            metric_card(
                "Average decision probability",
                f"{average_probability:.1%}",
                "Probability assigned to the actual referee call"
            )
        ], layout=widgets.Layout(gap="12px")))

    with table_output:
        clear_output(wait=True)

        if priority_df.empty:
            print("No incidents meet the current surprise threshold.")
            return

        table_df = priority_df[
            [
                "match_label",
                "minute",
                "player",
                "referee",
                "referee_call",
                "model_pred",
                "actual_probability",
                "surprise"
            ]
        ].head(25).copy()

        table_df.columns = [
            "Match",
            "Minute",
            "Player",
            "Referee",
            "Actual call",
            "Model prediction",
            "Chance of referee's call",
            "Surprise score"
        ]

        display(
            table_df.style.format({
                "Chance of referee's call": "{:.1%}",
                "Surprise score": "{:.2f}"
            })
        )


for control in [
    match_dropdown,
    referee_dropdown,
    decision_dropdown,
    surprise_slider
]:
    control.observe(update_your_dashboard, names="value")


#display(widgets.HTML("<h3>Dashboard summary</h3>"))
#display(summary_output)

#display(widgets.HTML("<h3>Ranked incidents for review</h3>"))
#display(table_output)

update_your_dashboard()

In [19]:
incident_dropdown = widgets.Dropdown(
    description="Review:",
    layout=widgets.Layout(width="700px")
)

incident_output = widgets.Output()


def show_selected_incident(change=None):
    priority_df = globals().get("current_priority_df", pd.DataFrame())

    if priority_df.empty or incident_dropdown.value is None:
        with incident_output:
            clear_output(wait=True)
            print("No high-priority incident is selected.")
        return

    selected_incident = priority_df[
        priority_df["id"] == incident_dropdown.value
    ].iloc[0]

    # This variable will be used by the pitch and probability charts later.
    globals()["selected_incident"] = selected_incident

    actual_call = selected_incident["referee_call"].replace("_", " ").title()
    predicted_call = selected_incident["model_pred"].replace("_", " ").title()

    with incident_output:
        clear_output(wait=True)

        display(widgets.HTML(f"""
        <div style="
            border-left: 4px solid #c84b3d;
           background: #fff4f2;
            color: #172235;
            padding: 14px;
            border-radius: 6px;
            width: 700px;
        ">
            <b>Selected incident: {selected_incident["home_team"]} vs
            {selected_incident["away_team"]} · {selected_incident["minute"]}'</b>
            <br>
            Player: {selected_incident["player"]} · Referee: {selected_incident["referee"]}
            <br>
            Actual referee call: <b>{actual_call}</b>
            <br>
            Model's most likely call: <b>{predicted_call}</b>
            <br>
            Probability of actual call: <b>{selected_incident["actual_probability"]:.1%}</b>
            · Surprise score: <b>{selected_incident["surprise"]:.2f}</b>
        </div>
        """))


def refresh_incident_options(change=None):
    priority_df = globals().get("current_priority_df", pd.DataFrame())

    options = [
        (
            f"{row['home_team']} vs {row['away_team']} | "
            f"{row['minute']}' | {row['referee_call'].replace('_', ' ').title()} | "
            f"surprise {row['surprise']:.2f}",
            row["id"]
        )
        for _, row in priority_df.head(100).iterrows()
    ]

    previous_value = incident_dropdown.value
    incident_dropdown.options = options

    available_ids = [incident_id for _, incident_id in options]

    if previous_value in available_ids:
        incident_dropdown.value = previous_value
    elif available_ids:
        incident_dropdown.value = available_ids[0]


incident_dropdown.observe(show_selected_incident, names="value")

for control in [
    match_dropdown,
    referee_dropdown,
    decision_dropdown,
    surprise_slider
]:
    control.observe(refresh_incident_options, names="value")


#display(widgets.HTML("<h3>Inspect one flagged incident</h3>"))
#display(incident_dropdown)
#display(incident_output)

refresh_incident_options()

In [20]:
reset_button = widgets.Button(
    description="Reset filters",
    icon="refresh",
    button_style=""
)

def reset_filters(button):
    match_dropdown.value = "All matches"
    referee_dropdown.value = "All referees"
    decision_dropdown.value = "all"
    surprise_slider.value = 1.0

reset_button.on_click(reset_filters)

#display(reset_button)

In [21]:
chart_output = widgets.Output()


def update_visualizations(change=None):
    filtered_df = globals().get("current_filtered_df", pd.DataFrame())
    priority_df = globals().get("current_priority_df", pd.DataFrame())
    selected = globals().get("selected_incident", None)

    with chart_output:
        clear_output(wait=True)

        if filtered_df.empty or priority_df.empty:
            print("No high-priority incidents match the current filters.")
            return

        # If the selected incident disappeared after filtering,
        # use the highest-priority remaining incident.
        if (
            selected is None
            or selected["id"] not in priority_df["id"].values
        ):
            selected = priority_df.iloc[0]
            globals()["selected_incident"] = selected

        display(make_probability_chart(selected))
        display(make_pitch_plot(filtered_df, selected))


# Update charts when the chosen incident changes.
incident_dropdown.observe(update_visualizations, names="value")

# Update charts when any filter changes.
for control in [
    match_dropdown,
    referee_dropdown,
    decision_dropdown,
    surprise_slider
]:
    control.observe(update_visualizations, names="value")


#display(widgets.HTML("<h3>Model decision visuals</h3>"))
#display(chart_output)

update_visualizations()

In [22]:
dashboard = widgets.VBox([
    widgets.HTML("""
        <h1 style="color: #172235; margin-bottom: 4px;">
            Main Dashboard
        </h1>
        <p style="color: #475569; margin-top: 0;">
            Referee incident review dashboard
        </p>
    """),

    widgets.HTML("<h3>Filters</h3>"),
    controls,
    reset_button,

    widgets.HTML("<hr><h3>Dashboard summary</h3>"),
    summary_output,

    widgets.HTML("<h3>Ranked incidents for review</h3>"),
    table_output,

    widgets.HTML("<h3>Inspect one flagged incident</h3>"),
    incident_dropdown,
    incident_output,

    widgets.HTML("<hr><h3>Model decision visuals</h3>"),
    chart_output,
], layout=widgets.Layout(
    width="100%",
    padding="20px",
    gap="12px"
))

display(dashboard)

In [23]:
from pathlib import Path
import html
import json
import ast
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output


def resolve_data_file(name):
    path = Path("data") / name

    if path.exists():
        return path

    csv_path = path.with_suffix(".csv")

    if csv_path.exists():
        return csv_path

    raise FileNotFoundError(
        f"Could not find {path} or {csv_path}"
    )


def find_column(dataframe, possible_names):
    for name in possible_names:
        if name in dataframe.columns:
            return name
    return None


def extract_incident_id(entry):
    # Audit entries may be dictionaries, JSON strings, or Python-style dict strings
    if isinstance(entry, str):
        try:
            entry = json.loads(entry)
        except (json.JSONDecodeError, TypeError):
            try:
                entry = ast.literal_eval(entry)
            except (ValueError, SyntaxError):
                return None

    if isinstance(entry, dict):
        # Check common names for the incident identifier
        for key in ["incident_id", "id"]:
            if key in entry and pd.notna(entry[key]):
                return str(entry[key]).strip()

        # Look inside nested dictionaries if needed
        for value in entry.values():
            if isinstance(value, dict):
                nested_id = extract_incident_id(value)
                if nested_id is not None:
                    return nested_id

    return None


# Load the three project files
REPLAY_PATH = resolve_data_file("replay_3754333")
EXPLANATIONS_PATH = resolve_data_file("explanations_3754333")
AUDIT_PATH = Path("data/audit_log_3754333.json")

if not AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find audit file: {AUDIT_PATH}"
    )

replay_df = pd.read_csv(REPLAY_PATH)
explanations_df = pd.read_csv(EXPLANATIONS_PATH)

try:
    audit_df = pd.read_json(AUDIT_PATH)
except ValueError:
    audit_df = pd.read_json(AUDIT_PATH, lines=True)


# Validate and standardize incident IDs
if "id" not in replay_df.columns:
    raise ValueError("replay_3754333.csv needs an `id` column.")

if "id" not in explanations_df.columns:
    raise ValueError("explanations_3754333.csv needs an `id` column.")

if "id" not in df.columns:
    raise ValueError("dashboard_incidents.csv needs an `id` column.")

replay_df["id"] = replay_df["id"].astype(str).str.strip()
explanations_df["id"] = explanations_df["id"].astype(str).str.strip()
df["id"] = df["id"].astype(str).str.strip()


# Extract incident IDs from the audit log's `entry` field
if "entry" not in audit_df.columns or "hash" not in audit_df.columns:
    raise ValueError(
        "Audit log must include `entry` and `hash` columns. "
        f"Found: {audit_df.columns.tolist()}"
    )

audit_df["id"] = audit_df["entry"].apply(extract_incident_id)

if audit_df["id"].isna().all():
    raise ValueError(
        "Could not find incident IDs inside the audit log entries."
    )

audit_df["id"] = audit_df["id"].astype(str).str.strip()

# Verify each row connects to the preceding hash in the log
if "prev_hash" in audit_df.columns:
    audit_df["chain_valid"] = (
        audit_df["prev_hash"].astype(str)
        == audit_df["hash"].shift(1).astype(str)
    )
    audit_df.loc[audit_df.index[0], "chain_valid"] = True
else:
    audit_df["chain_valid"] = audit_df["hash"].notna()

audit_for_merge = audit_df[
    ["id", "hash", "chain_valid"]
].drop_duplicates("id").rename(
    columns={"hash": "audit_hash"}
)


# Find the replay timing and review-flag columns
time_column = find_column(
    replay_df,
    ["ms", "delay_ms", "elapsed_ms", "duration_ms", "wait_ms"]
)

flag_column = find_column(
    replay_df,
    ["flag", "flagged", "is_flagged", "review_flag"]
)

if time_column is None:
    raise ValueError(
        f"Cannot find the timing column. Found: "
        f"{replay_df.columns.tolist()}"
    )

if flag_column is None:
    raise ValueError(
        f"Cannot find the flag column. Found: "
        f"{replay_df.columns.tolist()}"
    )


# Add model details to each replay event
model_columns = [
    "id", "minute", "second", "player", "team",
    "home_team", "away_team", "referee",
    "referee_call", "model_pred", "surprise"
]

model_columns = [
    column for column in model_columns
    if column in df.columns
]

timeline_df = replay_df.merge(
    df[model_columns].drop_duplicates("id"),
    on="id",
    how="left",
    suffixes=("_replay", "")
)


# Add written explanations for flagged incidents
explanation_column = find_column(
    explanations_df,
    ["explanation", "written_explanation", "reason", "text"]
)

if explanation_column is not None:
    timeline_df = timeline_df.merge(
        explanations_df[["id", explanation_column]].drop_duplicates("id"),
        on="id",
        how="left"
    ).rename(columns={explanation_column: "explanation"})
else:
    timeline_df["explanation"] = ""


# Add audit hash and audit-chain status
timeline_df = timeline_df.merge(
    audit_for_merge,
    on="id",
    how="left"
)


# Use replay surprise first; if it is blank, use dashboard/model surprise
surprise_values = pd.Series(
    pd.NA,
    index=timeline_df.index,
    dtype="Float64"
)

for column in ["surprise_replay", "surprise"]:
    if column in timeline_df.columns:
        numeric_values = pd.to_numeric(
            timeline_df[column],
            errors="coerce"
        )
        surprise_values = surprise_values.combine_first(numeric_values)

timeline_df["surprise_display"] = surprise_values


# Prepare timing and flags for the replay
timeline_df["_delay_ms"] = pd.to_numeric(
    timeline_df[time_column],
    errors="coerce"
).fillna(1000)

timeline_df["_is_flagged"] = (
    timeline_df[flag_column]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes", "y", "flagged"])
)

print(f"Loaded {len(timeline_df)} replay events")
print(f"Flagged events: {timeline_df['_is_flagged'].sum()}")
print(f"Events with audit hashes: {timeline_df['audit_hash'].notna().sum()}")

Loaded 61 replay events
Flagged events: 5
Events with audit hashes: 61


In [28]:
replay_index = widgets.IntSlider(
    value=0,
    min=0,
    max=len(timeline_df) - 1,
    step=1,
    description="Incident:",
    continuous_update=False,
    layout=widgets.Layout(width="500px")
)

speed_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=8.0,
    step=0.5,
    description="Speed:",
    readout_format=".1f",
    layout=widgets.Layout(width="300px")
)

# This is the actual Play/Pause button.
play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(timeline_df) - 1,
    step=1,
    interval=600
)

reset_replay_button = widgets.Button(
    description="Restart replay",
    icon="refresh"
)

replay_status = widgets.HTML()
replay_output = widgets.Output()


def get_value(row, possible_columns, default="Not available"):
    for column in possible_columns:
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return default


def render_replay(change=None):
    index = replay_index.value
    event = timeline_df.iloc[index]

    scoring_ms = pd.to_numeric(
        event.get(time_column),
        errors="coerce"
    )
    
    scoring_ms_text = (
        f"{scoring_ms:.1f} ms"
        if pd.notna(scoring_ms)
        else "Not available"
    )

    minute = get_value(event, ["minute", "minute_replay"], "?")
    second = get_value(event, ["second", "second_replay"], 0)
    player = get_value(event, ["player", "player_replay"], "Unknown player")
    team = get_value(event, ["team", "team_replay"], "Unknown team")
    call = get_value(
        event,
        ["referee_call", "referee_call_replay"],
        "Incident"
    )

    surprise = event.get("surprise_display")

    flagged_so_far = int(
        timeline_df.iloc[:index + 1]["_is_flagged"].sum()
    )

    is_flagged = bool(event["_is_flagged"])

    status_text = (
        "FLAGGED FOR REVIEW"
        if is_flagged
        else "Normal incident"
    )

    status_color = "#b42318" if is_flagged else "#267a45"

    replay_status.value = f"""
    <div style="
        padding: 12px;
        border-radius: 8px;
        background: #ffffff;
        color: #172235;
        border: 1px solid #cbd5e1;
    ">
        <b>Newcastle vs Spurs — live replay</b><br>
        Event {index + 1} of {len(timeline_df)}
        · Flags so far: {flagged_so_far}
        · <span style="color: {status_color}; font-weight: 700;">
            {status_text}
          </span>
    </div>
    """

    explanation = event.get("explanation", "")

    if pd.isna(explanation) or not str(explanation).strip():
        explanation = "Typical call for this context. No review needed"

    surprise_text = (
        f"{float(surprise):.2f}"
        if surprise is not None and pd.notna(surprise)
        else "Not available"
    )

    recent_events = timeline_df.iloc[max(0, index - 4):index + 1]

    feed_rows = ""

    for _, recent in recent_events.iterrows():
        recent_minute = get_value(
            recent,
            ["minute", "minute_replay"],
            "?"
        )

        recent_player = get_value(
            recent,
            ["player", "player_replay"],
            "Unknown player"
        )

        recent_call = get_value(
            recent,
            ["referee_call", "referee_call_replay"],
            "Incident"
        )

        recent_status = (
            "Flagged"
            if recent["_is_flagged"]
            else "Normal"
        )

        feed_rows += f"""
        <tr>
            <td>{html.escape(str(recent_minute))}'</td>
            <td>{html.escape(str(recent_player))}</td>
            <td>{html.escape(str(recent_call).replace("_", " ").title())}</td>
            <td>{recent_status}</td>
        </tr>
        """

    with replay_output:
        clear_output(wait=True)

        display(widgets.HTML(f"""
        <div style="
            margin-top: 10px;
            padding: 16px;
            border-left: 5px solid {status_color};
            border-radius: 8px;
            background: #f8fafc;
            color: #172235;
        ">
            <div style="font-size: 20px; font-weight: 700;">
                {html.escape(str(player))} · {minute}' {second}"
            </div>
            <div style="margin-top: 6px;">
                Team: <b>{html.escape(str(team))}</b>
                · Referee decision:
                <b>{html.escape(str(call).replace("_", " ").title())}</b>
            </div>
            <div style="margin-top: 6px;">
                Surprise score: <b>{surprise_text}</b>
                <br>Model scoring time:<b> {scoring_ms_text}</b>
            </div>
            <div style="margin-top: 12px; color: #475569;">
                <b>Assessor explanation:</b><br>
                {html.escape(str(explanation))}
            </div>
        </div>

        <h4 style="color: #172235;">Recent replay events</h4>

        <table style="
            border-collapse: collapse;
            width: 100%;
            color: #172235;
            background: #ffffff;
        ">
            <tr style="background: #eaf1f8;">
                <th style="padding: 7px; text-align: left;">Minute</th>
                <th style="padding: 7px; text-align: left;">Player</th>
                <th style="padding: 7px; text-align: left;">Decision</th>
                <th style="padding: 7px; text-align: left;">Status</th>
            </tr>
            {feed_rows}
        </table>
        """))


def update_replay_speed(change=None):
    # Deliberately use a consistent visible pace.
    # This makes the speed control clearly work for the demo.
    play_widget.interval = int(
        max(100, min(1400 / speed_slider.value, 3000))
    )


def restart_replay(button):
    play_widget.value = 0
    replay_index.value = 0


# IMPORTANT: `link`, not `jslink`.
# This sends every replay step back to Python so render_replay runs.
replay_link = widgets.link(
    (play_widget, "value"),
    (replay_index, "value")
)

replay_index.observe(render_replay, names="value")
speed_slider.observe(update_replay_speed, names="value")
reset_replay_button.on_click(restart_replay)

replay_controls = widgets.HBox([
    play_widget,
    reset_replay_button,
    replay_index,
    speed_slider
], layout=widgets.Layout(
    flex_flow="row wrap",
    gap="10px"
))

replay_section = widgets.VBox([
    widgets.HTML("<hr><h2>Live match replay</h2>"),
    replay_controls,
    replay_status,
    replay_output
], layout=widgets.Layout(
    width="100%",
    padding="8px 0",
    gap="10px"
))

# Prevent duplicate replay sections if this cell is rerun.
replay_section._is_live_replay = True

dashboard.children = tuple(
    child for child in dashboard.children
    if not getattr(child, "_is_live_replay", False)
) + (replay_section,)

update_replay_speed()
render_replay()

In [25]:
import hashlib, json

def make_hash(entry, prev_hash):
    text = json.dumps(entry, sort_keys=True) + prev_hash
    return hashlib.sha256(text.encode()).hexdigest()

def verify(log):
    prev = "0" * 64
    for i, rec in enumerate(log):
        if rec["prev_hash"] != prev or make_hash(rec["entry"], prev) != rec["hash"]:
            return False, i
        prev = rec["hash"]
    return True, None

audit = json.load(open("data/audit_log_3754333.json"))
chain_ok, broken_at = verify(audit)
hash_by_id = {rec["entry"]["incident_id"]: rec["hash"] for rec in audit}

audit_banner = widgets.HTML(f"""
<div style="padding:10px;border-radius:8px;background:{'#ecfdf3' if chain_ok else '#fef3f2'};
color:#172235;border:1px solid {'#267a45' if chain_ok else '#b42318'};">
<b>Secure audit log:</b> {len(audit)} incidents hash-chained ·
{'chain valid ✓' if chain_ok else f'chain broken at entry {broken_at} ✗'}
</div>""")
audit_line = widgets.HTML()

def render_audit(change=None):
    event = timeline_df.iloc[replay_index.value]
    h = hash_by_id.get(str(event["id"]))
    audit_line.value = (
        f"<div style='color:#172235;background:#f8fafc;border:1px solid #cbd5e1;border-radius:8px;padding:8px 12px;font-family:monospace;'>"
        f"Scored in {float(event['_delay_ms']):.1f} ms on LinuxONE · "
        f"logged to audit chain: {h[:12] + '…' if h else 'not found'}</div>")

replay_index.observe(render_audit, names="value")
replay_section.children = (replay_section.children[0], audit_banner, replay_controls, replay_status, audit_line, replay_output)
render_audit()